# 🎯 TIKI ABSA Training — Google Colab
**Aspect-Based Sentiment Analysis cho sản phẩm trẻ sơ sinh (Tiki)**

---
## 📋 Quy trình
1. ⚙️ Setup môi trường
2. 📂 Chuẩn bị dữ liệu
3. 🤖 Train PhoBERT
4. 🧠 Train BiLSTM-CRF
5. 📊 Train SVM + TF-IDF (baseline, không cần GPU)
6. 📈 So sánh kết quả 3 mô hình

---
⚠️ **Lưu ý Colab mất kết nối**: Mỗi bước đều tự động lưu checkpoint.  
Nếu bị ngắt → **Runtime > Run all** → sẽ tự resume từ checkpoint mới nhất.

💡 **SVM không cần GPU** — có thể chạy trên CPU runtime bình thường.

---
## CELL 1 — Mount Google Drive & Clone/Upload project

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 1: Mount Drive
# ══════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted')

---
## CELL 1b — Khôi phục checkpoint từ Drive (chạy khi mất kết nối)

> **Chạy cell này ngay sau khi mount Drive nếu:**
> - Colab vừa bị mất kết nối / runtime reset
> - best_model.pt bị xóa nhầm
> - Muốn kiểm tra trạng thái checkpoint hiện tại

In [ ]:
# ==============================================================
# CELL 1b: RECOVERY - Khoi phuc checkpoint + sua best_model
# ==============================================================
import os, shutil, torch

DRIVE_PROJECT = '/content/drive/MyDrive/tiki_absa'
COLAB_PROJECT = '/content/tiki'

def restore_and_clean(drive_ckpt, local_ckpt):
    """
    Copy checkpoint tu Drive ve local, kiem tra tung file,
    xoa file corrupt o CA HAI noi (local + Drive).
    Tra ve danh sach epoch hop le.
    """
    if not os.path.exists(drive_ckpt):
        return []

    os.makedirs(local_ckpt, exist_ok=True)
    shutil.copytree(drive_ckpt, local_ckpt, dirs_exist_ok=True)

    pts = [f for f in os.listdir(local_ckpt)
           if f.startswith('epoch_') and f.endswith('.pt')]
    good, bad = [], []

    for fname in pts:
        local_path = os.path.join(local_ckpt, fname)
        try:
            torch.load(local_path, map_location='cpu')
            good.append(fname)
        except Exception as e:
            bad.append(fname)
            os.remove(local_path)
            drive_path = os.path.join(drive_ckpt, fname)
            if os.path.exists(drive_path):
                os.remove(drive_path)
            print(f'  [X] {fname} CORRUPT ({type(e).__name__}) - da xoa local + Drive')

    if good:
        print(f'  [OK] Checkpoints hop le: {sorted(good)}')
    return good


def try_load_best(ckpt_dir):
    """Load checkpoint tot nhat (epoch lon nhat con hop le)."""
    if not os.path.exists(ckpt_dir):
        return None, 0
    pts = [f for f in os.listdir(ckpt_dir)
           if f.startswith('epoch_') and f.endswith('.pt')]
    if not pts:
        return None, 0
    epochs = sorted([int(p.replace('epoch_','').replace('.pt','')) for p in pts], reverse=True)
    for ep in epochs:
        path = os.path.join(ckpt_dir, f'epoch_{ep}.pt')
        try:
            state = torch.load(path, map_location='cpu')
            return state, ep
        except Exception:
            os.remove(path)
    return None, 0


# ── Buoc 1: Restore + clean checkpoints (chi phobert va bilstm) ──────
print('=' * 55)
print('  RECOVERY: Restore va kiem tra checkpoint')
print('=' * 55)

for model_name in ['phobert', 'bilstm']:
    print(f'\n[{model_name}]')
    drive_ckpt  = f'{DRIVE_PROJECT}/checkpoints/{model_name}'
    local_ckpt  = f'{COLAB_PROJECT}/checkpoints/{model_name}'
    drive_model = f'{DRIVE_PROJECT}/models/{model_name}'
    local_model = f'{COLAB_PROJECT}/models/{model_name}'

    good = restore_and_clean(drive_ckpt, local_ckpt)
    if not good:
        print(f'  Khong co checkpoint hop le tren Drive')

    if os.path.exists(drive_model):
        os.makedirs(local_model, exist_ok=True)
        shutil.copytree(drive_model, local_model, dirs_exist_ok=True)
        print(f'  models/{model_name}/ restored: {os.listdir(local_model)}')


# ── Buoc 2: Restore SVM model (.pkl) neu co ──────────────────────────
print('\n[svm]')
drive_svm = f'{DRIVE_PROJECT}/models/svm'
local_svm = f'{COLAB_PROJECT}/models/svm'
if os.path.exists(drive_svm):
    os.makedirs(local_svm, exist_ok=True)
    shutil.copytree(drive_svm, local_svm, dirs_exist_ok=True)
    print(f'  models/svm/ restored: {os.listdir(local_svm)}')
else:
    print('  Khong co SVM model tren Drive (chua train)')


# ── Buoc 3: Tao lai best_model neu thieu (phobert + bilstm) ──────────
print('\n--- Kiem tra best_model ---')
for model_name in ['phobert', 'bilstm']:
    model_dir = f'{COLAB_PROJECT}/models/{model_name}'
    ckpt_dir  = f'{COLAB_PROJECT}/checkpoints/{model_name}'
    best_path = f'{model_dir}/best_model.pt'

    if os.path.exists(best_path):
        try:
            ckpt  = torch.load(best_path, map_location='cpu')
            val_m = ckpt.get('val_metrics', {})
            ep    = ckpt.get('epoch', '?')
            print(f'[{model_name}] best_model.pt OK '
                  f'(epoch {ep}, avg_f1={val_m.get("avg_f1", 0):.4f})')
            continue
        except Exception as e:
            print(f'[{model_name}] best_model.pt CORRUPT ({type(e).__name__}), tao lai...')
            os.remove(best_path)

    print(f'[{model_name}] best_model.pt khong ton tai -> tim checkpoint...')
    state, ep = try_load_best(ckpt_dir)
    if state is None:
        print(f'  -> Khong co checkpoint hop le. Can train lai tu dau.')
        continue
    os.makedirs(model_dir, exist_ok=True)
    torch.save({
        'epoch':       ep,
        'model_state': state['model_state'],
        'val_metrics': state.get('val_metrics', {}),
    }, best_path)
    val_m = state.get('val_metrics', {})
    print(f'  -> Tao lai best_model.pt tu epoch {ep} '
          f'(avg_f1={val_m.get("avg_f1", 0):.4f})')
    print(f'     => Cell train se resume tu epoch {ep + 1}')

    drive_model = f'{DRIVE_PROJECT}/models/{model_name}'
    os.makedirs(drive_model, exist_ok=True)
    shutil.copy2(best_path, os.path.join(drive_model, 'best_model.pt'))
    print(f'  -> Da sync best_model.pt len Drive')

print()
print('XONG. Chay Cell 6/8/9 de tiep tuc train.')

---
## CELL 2 — Tạo cấu trúc thư mục & copy files từ Drive

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 2: Setup thư mục dự án
# ══════════════════════════════════════════════════════
import os, shutil

# ─── CẤU HÌNH: thay đổi nếu cần ───────────────────────
DRIVE_PROJECT = '/content/drive/MyDrive/tiki_absa'  # thư mục trên Drive
COLAB_PROJECT = '/content/tiki'                      # thư mục làm việc
# ────────────────────────────────────────────────────────

os.makedirs(COLAB_PROJECT, exist_ok=True)

# Copy toàn bộ project từ Drive về Colab workspace
if os.path.exists(DRIVE_PROJECT):
    print(f'Copying {DRIVE_PROJECT} → {COLAB_PROJECT}...')
    if os.path.exists(f'{DRIVE_PROJECT}/src'):
        shutil.copytree(f'{DRIVE_PROJECT}/src', f'{COLAB_PROJECT}/src',
                        dirs_exist_ok=True)
    if os.path.exists(f'{DRIVE_PROJECT}/data'):
        shutil.copytree(f'{DRIVE_PROJECT}/data', f'{COLAB_PROJECT}/data',
                        dirs_exist_ok=True)
    for f in ['requirements_new.txt', 'requirements.txt']:
        src = f'{DRIVE_PROJECT}/{f}'
        if os.path.exists(src):
            shutil.copy2(src, f'{COLAB_PROJECT}/{f}')
    for folder in ['checkpoints', 'models', 'results']:
        src = f'{DRIVE_PROJECT}/{folder}'
        if os.path.exists(src):
            shutil.copytree(src, f'{COLAB_PROJECT}/{folder}', dirs_exist_ok=True)
            print(f'  ✅ Restored {folder}/')
    print('✅ Copy hoàn tất!')
else:
    print(f'⚠️  {DRIVE_PROJECT} chưa tồn tại trên Drive.')
    print('   Hãy upload project lên Drive trước (xem hướng dẫn bên dưới).')

# Tạo tất cả thư mục cần thiết
dirs_needed = [
    f'{COLAB_PROJECT}/data/processed',
    f'{COLAB_PROJECT}/data/training',
    f'{COLAB_PROJECT}/data/raw',
    f'{COLAB_PROJECT}/checkpoints/bilstm',
    f'{COLAB_PROJECT}/checkpoints/phobert',
    f'{COLAB_PROJECT}/models/bilstm',
    f'{COLAB_PROJECT}/models/phobert',
    f'{COLAB_PROJECT}/models/svm',
    f'{COLAB_PROJECT}/results',
    f'{COLAB_PROJECT}/src/training',
]
for d in dirs_needed:
    os.makedirs(d, exist_ok=True)

# Set working directory
os.chdir(COLAB_PROJECT)
import sys
sys.path.insert(0, COLAB_PROJECT)

print(f'\n✅ Working directory: {os.getcwd()}')
print('\nCấu trúc thư mục:')
for d in ['src/training', 'data/processed', 'data/training',
          'checkpoints', 'models', 'results']:
    exists = '✅' if os.path.exists(d) else '❌'
    print(f'  {exists} {d}/')

---
## CELL 3 — Cài đặt thư viện

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 3: Cai dependencies
# ══════════════════════════════════════════════════════
import subprocess, sys

packages = [
    "transformers>=4.36.0",
    "torch>=2.1.0",
    "torchcrf>=1.1.0",
    "scikit-learn>=1.3.0",
    "numpy>=1.24.0",
    "pandas>=2.0.0",
    "accelerate>=0.26.0",
    "sentencepiece>=0.1.99",
]

print("Dang cai packages (co the mat 1-2 phut)...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + packages,
    capture_output=True, text=True
)
if result.returncode == 0:
    print("✅ Cai dat thanh cong!")
else:
    print("Co loi:", result.stderr[-300:])

import torch, transformers, sklearn
print(f"PyTorch       : {torch.__version__}")
print(f"Transformers  : {transformers.__version__}")
print(f"scikit-learn  : {sklearn.__version__}")
print(f"CUDA          : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
    print(f"VRAM          : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

---
## CELL 4 — Kiểm tra dữ liệu annotation

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 4: Kiểm tra dữ liệu
# ══════════════════════════════════════════════════════
import json, os
from collections import Counter

data_file = 'data/processed/asqp_annotated.jsonl'

if not os.path.exists(data_file):
    print(f'❌ Không tìm thấy {data_file}')
    print('   Hãy upload file asqp_annotated.jsonl vào data/processed/')
else:
    total = 0
    cat_counts = Counter()
    sent_counts = Counter()
    with open(data_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            d = json.loads(line)
            total += 1
            for q in d.get('quadruples', []):
                cat_counts[q.get('aspect_category', 'UNK')] += 1
                sent_counts[q.get('sentiment', 'UNK')] += 1

    print(f'✅ File: {data_file}')
    print(f'   Reviews: {total:,}')
    print(f'   Sentiment: {dict(sent_counts)}')
    print(f'   Categories: {len(cat_counts)}')
    print(f'   Top 5: {dict(cat_counts.most_common(5))}')

---
## CELL 5 — Chuẩn bị dữ liệu (chạy 1 lần)

> Output của bước này được **dùng chung cho cả 3 mô hình** (PhoBERT, BiLSTM, SVM).  
> SVM dùng lại `phobert_*.csv` — **không cần prepare riêng**.

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 5: prepare_data.py
# Chỉ cần chạy 1 lần. Bỏ qua nếu data/training/ đã có files.
# Output dùng chung cho PhoBERT, BiLSTM VÀ SVM.
# ══════════════════════════════════════════════════════
import os

already_done = all(os.path.exists(f'data/training/{f}') for f in [
    'phobert_train.csv', 'phobert_val.csv', 'phobert_test.csv',
    'bilstm_train.txt', 'vocab.json'
])

if already_done:
    print('✅ Dữ liệu đã được chuẩn bị trước đó. Bỏ qua bước này.')
    import subprocess
    result = subprocess.run(['wc', '-l', 'data/training/phobert_train.csv'],
                           capture_output=True, text=True)
    print(f'   phobert_train.csv: {result.stdout.strip()} dòng')
    print('   (SVM sẽ dùng lại phobert_*.csv — không cần prepare riêng)')
else:
    print('Đang chuẩn bị dữ liệu...')
    import subprocess
    result = subprocess.run(
        ['python', 'src/training/prepare_data.py'],
        capture_output=False, text=True
    )
    if result.returncode == 0:
        print('\n✅ Chuẩn bị dữ liệu hoàn tất!')
        print('   Files tạo ra: bilstm_*.txt, phobert_*.csv, vocab.json')
        print('   → PhoBERT, BiLSTM, SVM đều dùng được ngay.')
    else:
        print('❌ Lỗi:', result.stderr)

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 5b: Sync data lên Drive (để không phải prepare lại)
# ══════════════════════════════════════════════════════
import shutil, os

DRIVE_PROJECT = '/content/drive/MyDrive/tiki_absa'
os.makedirs(f'{DRIVE_PROJECT}/data/training', exist_ok=True)

shutil.copytree('data/training', f'{DRIVE_PROJECT}/data/training',
                dirs_exist_ok=True)
print(f'✅ data/training synced → {DRIVE_PROJECT}/data/training')

---
## CELL 6 — Train PhoBERT ⭐ (khuyến nghị chạy trước)

> **Ước tính thời gian:**
> - T4 GPU (free): ~25-35 phút/epoch × 15 epochs = ~7-8 giờ
> - A100 GPU (Pro): ~7-10 phút/epoch × 15 epochs = ~2-3 giờ
>
> ⚠️ Nếu Colab mất kết nối giữa chừng → **Runtime > Run all** → tự resume từ epoch cuối

In [ ]:
import os, shutil, torch

DRIVE_PROJECT = '/content/drive/MyDrive/tiki_absa'
COLAB_PROJECT = '/content/tiki'

def clean_checkpoints(ckpt_dir, drive_ckpt_dir):
    """Quet tat ca epoch_N.pt, xoa file corrupt o ca local lan Drive."""
    if not os.path.exists(ckpt_dir):
        return []
    pts = [f for f in os.listdir(ckpt_dir)
           if f.startswith('epoch_') and f.endswith('.pt')]
    good = []
    for fname in sorted(pts, reverse=True):
        local_path = os.path.join(ckpt_dir, fname)
        try:
            torch.load(local_path, map_location='cpu')
            good.append(fname)
        except Exception as e:
            os.remove(local_path)
            drive_path = os.path.join(drive_ckpt_dir, fname)
            if os.path.exists(drive_path):
                os.remove(drive_path)
            print(f'  [X] {fname} corrupt ({type(e).__name__}) - da xoa local + Drive')
    return good

def is_training_complete(ckpt_dir, total_epochs, patience):
    if not os.path.exists(ckpt_dir):
        return False, 0
    pts = [f for f in os.listdir(ckpt_dir)
           if f.startswith('epoch_') and f.endswith('.pt')]
    if not pts:
        return False, 0
    epochs = sorted([int(p.replace('epoch_','').replace('.pt','')) for p in pts])
    latest = epochs[-1]
    if latest >= total_epochs:
        return True, latest
    path = os.path.join(ckpt_dir, f'epoch_{latest}.pt')
    try:
        ckpt = torch.load(path, map_location='cpu')
        if ckpt.get('no_improve', 0) >= patience:
            return True, latest
    except Exception:
        pass
    return False, latest

TOTAL_EPOCHS   = 15
PATIENCE       = 5
CHECKPOINT_DIR = 'checkpoints/phobert'
MODEL_DIR      = 'models/phobert'
DRIVE_CKPT     = f'{DRIVE_PROJECT}/checkpoints/phobert'

good = clean_checkpoints(CHECKPOINT_DIR, DRIVE_CKPT)
if good:
    epochs_good = sorted([int(f.replace('epoch_','').replace('.pt','')) for f in good])
    print(f'[PhoBERT] Checkpoints hop le: epoch {epochs_good}')
else:
    print('[PhoBERT] Khong co checkpoint hop le')

done, latest_ep = is_training_complete(CHECKPOINT_DIR, TOTAL_EPOCHS, PATIENCE)

if done and os.path.exists(f'{MODEL_DIR}/best_model.pt'):
    try:
        ckpt  = torch.load(f'{MODEL_DIR}/best_model.pt', map_location='cpu')
        val_m = ckpt.get('val_metrics', {})
        print(f'Training PhoBERT HOAN THANH (epoch {ckpt.get("epoch","?")}): '
              f'AD-F1={val_m.get("ad_f1",0):.4f} | AP-F1={val_m.get("ap_f1",0):.4f}')
        print('  De train lai: xoa checkpoints/phobert/ va models/phobert/')
    except Exception:
        done = False

if not done:
    if latest_ep > 0:
        print(f'Resume PhoBERT tu epoch {latest_ep} -> con {TOTAL_EPOCHS - latest_ep} epoch nua')
    else:
        print('Bat dau train PhoBERT tu epoch 1...')
    get_ipython().system('python src/training/train_phobert.py')

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 6b: Sync PhoBERT checkpoints lên Drive
# ══════════════════════════════════════════════════════
import shutil, os
DRIVE_PROJECT = '/content/drive/MyDrive/tiki_absa'

for folder in ['checkpoints/phobert', 'models/phobert']:
    if os.path.exists(folder):
        dst = f'{DRIVE_PROJECT}/{folder}'
        os.makedirs(dst, exist_ok=True)
        shutil.copytree(folder, dst, dirs_exist_ok=True)
        print(f'✅ {folder} → Drive')

if os.path.exists('results'):
    shutil.copytree('results', f'{DRIVE_PROJECT}/results', dirs_exist_ok=True)
    print('✅ results/ → Drive')

---
## CELL 7 — Train BiLSTM-CRF (baseline)

> **Ước tính thời gian:** ~5-10 phút/epoch × 25 epochs = ~2-4 giờ (CPU được)

In [ ]:
import os, shutil, torch

DRIVE_PROJECT = '/content/drive/MyDrive/tiki_absa'
COLAB_PROJECT = '/content/tiki'

def clean_checkpoints(ckpt_dir, drive_ckpt_dir):
    if not os.path.exists(ckpt_dir):
        return []
    pts = [f for f in os.listdir(ckpt_dir)
           if f.startswith('epoch_') and f.endswith('.pt')]
    good = []
    for fname in sorted(pts, reverse=True):
        local_path = os.path.join(ckpt_dir, fname)
        try:
            torch.load(local_path, map_location='cpu')
            good.append(fname)
        except Exception as e:
            os.remove(local_path)
            drive_path = os.path.join(drive_ckpt_dir, fname)
            if os.path.exists(drive_path):
                os.remove(drive_path)
            print(f'  [X] {fname} corrupt ({type(e).__name__}) - da xoa local + Drive')
    return good

def is_training_complete(ckpt_dir, total_epochs, patience):
    if not os.path.exists(ckpt_dir):
        return False, 0
    pts = [f for f in os.listdir(ckpt_dir)
           if f.startswith('epoch_') and f.endswith('.pt')]
    if not pts:
        return False, 0
    epochs = sorted([int(p.replace('epoch_','').replace('.pt','')) for p in pts])
    latest = epochs[-1]
    if latest >= total_epochs:
        return True, latest
    path = os.path.join(ckpt_dir, f'epoch_{latest}.pt')
    try:
        ckpt = torch.load(path, map_location='cpu')
        if ckpt.get('no_improve', 0) >= patience:
            return True, latest
    except Exception:
        pass
    return False, latest

TOTAL_EPOCHS   = 25
PATIENCE       = 6
CHECKPOINT_DIR = 'checkpoints/bilstm'
MODEL_DIR      = 'models/bilstm'
DRIVE_CKPT     = f'{DRIVE_PROJECT}/checkpoints/bilstm'

good = clean_checkpoints(CHECKPOINT_DIR, DRIVE_CKPT)
if good:
    epochs_good = sorted([int(f.replace('epoch_','').replace('.pt','')) for f in good])
    print(f'[BiLSTM] Checkpoints hop le: epoch {epochs_good}')
else:
    print('[BiLSTM] Khong co checkpoint hop le')

done, latest_ep = is_training_complete(CHECKPOINT_DIR, TOTAL_EPOCHS, PATIENCE)

if done and os.path.exists(f'{MODEL_DIR}/best_model.pt'):
    try:
        ckpt  = torch.load(f'{MODEL_DIR}/best_model.pt', map_location='cpu')
        val_m = ckpt.get('val_metrics', {})
        print(f'Training BiLSTM HOAN THANH (epoch {ckpt.get("epoch","?")}): '
              f'AD-F1={val_m.get("ad_f1",0):.4f} | AP-F1={val_m.get("ap_f1",0):.4f}')
        print('  De train lai: xoa checkpoints/bilstm/ va models/bilstm/')
    except Exception:
        done = False

if not done:
    if latest_ep > 0:
        print(f'Resume BiLSTM tu epoch {latest_ep} -> con {TOTAL_EPOCHS - latest_ep} epoch nua')
    else:
        print('Bat dau train BiLSTM tu epoch 1...')
    get_ipython().system('python src/training/train_bilstm.py')

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 7b: Sync BiLSTM lên Drive
# ══════════════════════════════════════════════════════
import shutil, os
DRIVE_PROJECT = '/content/drive/MyDrive/tiki_absa'
for folder in ['checkpoints/bilstm', 'models/bilstm', 'results']:
    if os.path.exists(folder):
        shutil.copytree(folder, f'{DRIVE_PROJECT}/{folder}', dirs_exist_ok=True)
        print(f'✅ {folder} → Drive')

---
## CELL 8 — Train SVM + TF-IDF 📊 (baseline, không cần GPU)

> **Ước tính thời gian:** ~30-60 giây (CPU)  
> **Không cần GPU** — có thể chạy trên bất kỳ runtime nào  
> **Dùng lại dữ liệu** từ `phobert_*.csv` — không cần prepare thêm
>
> 💡 SVM lưu model dưới dạng `.pkl`, không có checkpoint (train 1 lần là xong)

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 8: Train SVM + TF-IDF
# ══════════════════════════════════════════════════════
import os

DRIVE_PROJECT = '/content/drive/MyDrive/tiki_absa'
MODEL_PATH    = 'models/svm/svm_model.pkl'

# Kiểm tra dữ liệu đầu vào
data_ok = all(os.path.exists(f'data/training/{f}') for f in [
    'phobert_train.csv', 'phobert_val.csv', 'phobert_test.csv'
])
if not data_ok:
    print('❌ Chưa có dữ liệu. Hãy chạy CELL 5 (prepare_data.py) trước.')
else:
    # Kiểm tra đã train chưa
    if os.path.exists(MODEL_PATH):
        import pickle, os
        try:
            with open(MODEL_PATH, 'rb') as f:
                state = pickle.load(f)
            cfg = state.get('config', {})
            print(f'✅ SVM model đã train:')
            print(f'   max_features : {cfg.get("max_features", "?")}')
            print(f'   ngram_range  : {cfg.get("ngram_range", "?")}')
            print(f'   svm_c        : {cfg.get("svm_c", "?")}')
            print('   Để train lại: xóa models/svm/svm_model.pkl')
        except Exception as e:
            print(f'⚠️ Model file bị lỗi ({e}), train lại...')
            os.remove(MODEL_PATH)
            get_ipython().system('python src/training/train_svm_tfidf.py')
    else:
        print('Bắt đầu train SVM + TF-IDF...')
        print('(Không cần GPU — chỉ mất ~30-60 giây)')
        get_ipython().system('python src/training/train_svm_tfidf.py')

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 8b: Sync SVM model lên Drive
# ══════════════════════════════════════════════════════
import shutil, os
DRIVE_PROJECT = '/content/drive/MyDrive/tiki_absa'

for folder in ['models/svm', 'results']:
    if os.path.exists(folder):
        dst = f'{DRIVE_PROJECT}/{folder}'
        os.makedirs(dst, exist_ok=True)
        shutil.copytree(folder, dst, dirs_exist_ok=True)
        print(f'✅ {folder} → Drive')

---
## CELL 9 — So sánh kết quả 3 mô hình

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 9: Xem kết quả đã lưu (từ results/*.json)
# ══════════════════════════════════════════════════════
import json, os

models_info = [
    ('PhoBERT',      'results/phobert_results.json'),
    ('BiLSTM-CRF',   'results/bilstm_results.json'),
    ('SVM+TF-IDF',   'results/svm_results.json'),
]

print('='*70)
print(f"  {'Model':<14} {'AD-Prec':>9} {'AD-Rec':>8} {'AD-F1':>8} "
      f"{'AP-Prec':>9} {'AP-Rec':>8} {'AP-F1':>8} {'Avg-F1':>8}")
print('  ' + '-'*68)

best_name, best_f1 = '', 0.0
for name, path in models_info:
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        t = data.get('test', data.get('best_val', {}))
        avg = t.get('avg_f1', 0)
        marker = ' ★' if avg > best_f1 else ''
        if avg > best_f1:
            best_f1, best_name = avg, name
        print(f"  {name:<14} "
              f"{t.get('ad_precision', 0):>9.4f} "
              f"{t.get('ad_recall', 0):>8.4f} "
              f"{t.get('ad_f1', 0):>8.4f} "
              f"{t.get('ap_precision', 0):>9.4f} "
              f"{t.get('ap_recall', 0):>8.4f} "
              f"{t.get('ap_f1', 0):>8.4f} "
              f"{avg:>8.4f}{marker}")
    else:
        print(f"  {name:<14} {'(chưa train)':>55}")

print('='*70)
if best_name:
    print(f"  ★ Best: {best_name} (Avg F1 = {best_f1:.4f})")

# Xem chi tiết từng model nếu muốn
print('\n--- Chi tiết SVM (nếu đã train) ---')
if os.path.exists('results/svm_results.json'):
    with open('results/svm_results.json') as f:
        svm_data = json.load(f)
    t = svm_data.get('test', {})
    cfg = svm_data.get('config', {})
    print(f"  Train time   : {svm_data.get('train_time', 0):.1f}s")
    print(f"  ngram_range  : {cfg.get('ngram_range')}")
    print(f"  max_features : {cfg.get('max_features')}")
    print(f"  Train samples: {cfg.get('train_samples')}")
    print(f"  Test samples : {cfg.get('test_samples')}")

---
## CELL 9b — Đánh giá chi tiết (evaluate.py)

> Chạy `evaluate.py` để tính lại metrics đầy đủ, in per-category breakdown và lưu báo cáo tổng hợp.

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 9b: evaluate.py — đánh giá đầy đủ tất cả model
# ══════════════════════════════════════════════════════
import os

# Đánh giá tất cả model đã train
# Thay --model all bằng --model bilstm / phobert / svm nếu muốn chạy riêng
get_ipython().system('python src/training/evaluate.py --model all')

# Xem báo cáo text
if os.path.exists('results/evaluation_report.txt'):
    print('\n' + '='*55)
    print('  BÁO CÁO TỔNG HỢP')
    print('='*55)
    with open('results/evaluation_report.txt', 'r', encoding='utf-8') as f:
        print(f.read())

---
## CELL 10 — Sync tất cả về Drive (chạy cuối cùng)

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 10: Full sync về Drive
# ══════════════════════════════════════════════════════
import shutil, os
DRIVE_PROJECT = '/content/drive/MyDrive/tiki_absa'
os.makedirs(DRIVE_PROJECT, exist_ok=True)

folders_to_sync = [
    'checkpoints',
    'models',
    'results',
    'data/training',
]

for folder in folders_to_sync:
    if os.path.exists(folder):
        dst = f'{DRIVE_PROJECT}/{folder}'
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copytree(folder, dst, dirs_exist_ok=True)
        size = sum(os.path.getsize(os.path.join(dp, f))
                   for dp, dn, fn in os.walk(folder) for f in fn)
        print(f'✅ {folder} → Drive ({size/1e6:.1f} MB)')

print('\n🎉 Tất cả đã được lưu lên Drive!')
print(f'   Location: {DRIVE_PROJECT}')

# Tóm tắt files quan trọng
print('\n📦 Files model quan trọng:')
important = [
    'models/phobert/best_model.pt',
    'models/bilstm/best_model.pt',
    'models/svm/svm_model.pkl',
]
for p in important:
    if os.path.exists(p):
        size = os.path.getsize(p)
        print(f'  ✅ {p} ({size/1e6:.1f} MB)')
    else:
        print(f'  ❌ {p} (chưa train)')

---
## CELL 11 — Test nhanh model (inference demo)

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 11: Quick inference test với PhoBERT
# ══════════════════════════════════════════════════════
import torch, sys, json, os
sys.path.insert(0, '/content/tiki')

CATEGORIES = [
    'PRODUCT#QUALITY','DELIVERY#SPEED','DELIVERY#PACKAGING',
    'PRICE#AFFORDABILITY','SELLER#SERVICE','PRODUCT#FUNCTION',
    'PRODUCT#COMFORT','PRODUCT#DESIGN','DELIVERY#ACCURACY',
    'PRODUCT#DURABILITY','PRODUCT#SAFETY','SELLER#AUTHENTICITY',
    'PRODUCT#MATERIAL','PRODUCT#SIZE','PRODUCT#VALUE',
    'PRICE#DISCOUNT','SELLER#RESPONSIVENESS'
]
SENT_NAMES = ['none', 'positive', 'neutral', 'negative']

test_reviews = [
    "bỉm mềm thấm hút tốt nhưng giá hơi đắt giao hàng nhanh",
    "sản phẩm kém chất lượng giao hàng chậm đóng gói cẩu thả",
    "chất lượng tuyệt vời đúng như mô tả giá hợp lý",
]

def print_results(name, results_per_review):
    print(f'\n=== INFERENCE DEMO — {name} ===\n')
    for review, results in zip(test_reviews, results_per_review):
        print(f'Input : {review}')
        if results:
            for r in results:
                print(f'       ✓ {r}')
        else:
            print('       (không phát hiện aspect)')
        print()

# ── PhoBERT ──────────────────────────────────────────
model_path = 'models/phobert/best_model.pt'
if not os.path.exists(model_path):
    print('⚠️ PhoBERT chưa train. Chạy Cell 6 trước.')
else:
    from transformers import AutoTokenizer
    from src.training.train_phobert import PhoBERTASQP, CFG

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tokenizer = AutoTokenizer.from_pretrained('vinai/phobert-base-v2')
    ckpt = torch.load(model_path, map_location=device)
    model = PhoBERTASQP(CFG).to(device)
    model.load_state_dict(ckpt['model_state'])
    model.eval()

    all_results = []
    with torch.no_grad():
        for review in test_reviews:
            enc = tokenizer(review, return_tensors='pt',
                           max_length=256, truncation=True,
                           padding='max_length').to(device)
            logits = model(enc['input_ids'], enc['attention_mask'])
            results = []
            for k, logit in enumerate(logits):
                pred = logit.argmax(-1).item()
                if pred != 0:
                    results.append(f"{CATEGORIES[k]} → {SENT_NAMES[pred]}")
            all_results.append(results)
    print_results('PhoBERT', all_results)

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 11b: Quick inference test với SVM + TF-IDF
# ══════════════════════════════════════════════════════
import sys, os
sys.path.insert(0, '/content/tiki')

model_path = 'models/svm/svm_model.pkl'
if not os.path.exists(model_path):
    print('⚠️ SVM chưa train. Chạy Cell 8 trước.')
else:
    from src.training.train_svm_tfidf import SVMTFIDFModel

    CATEGORIES = [
        'PRODUCT#QUALITY','DELIVERY#SPEED','DELIVERY#PACKAGING',
        'PRICE#AFFORDABILITY','SELLER#SERVICE','PRODUCT#FUNCTION',
        'PRODUCT#COMFORT','PRODUCT#DESIGN','DELIVERY#ACCURACY',
        'PRODUCT#DURABILITY','PRODUCT#SAFETY','SELLER#AUTHENTICITY',
        'PRODUCT#MATERIAL','PRODUCT#SIZE','PRODUCT#VALUE',
        'PRICE#DISCOUNT','SELLER#RESPONSIVENESS'
    ]
    SENT_NAMES = ['none', 'positive', 'neutral', 'negative']

    test_reviews = [
        "bỉm mềm thấm hút tốt nhưng giá hơi đắt giao hàng nhanh",
        "sản phẩm kém chất lượng giao hàng chậm đóng gói cẩu thả",
        "chất lượng tuyệt vời đúng như mô tả giá hợp lý",
    ]

    svm_model = SVMTFIDFModel.load(model_path)
    preds = svm_model.predict(test_reviews)  # [3, 17]

    print('=== INFERENCE DEMO — SVM + TF-IDF ===\n')
    for review, pred_row in zip(test_reviews, preds):
        print(f'Input : {review}')
        results = [
            f"{CATEGORIES[k]} → {SENT_NAMES[pred_row[k]]}"
            for k in range(len(CATEGORIES)) if pred_row[k] != 0
        ]
        if results:
            for r in results:
                print(f'       ✓ {r}')
        else:
            print('       (không phát hiện aspect)')
        print()